In [ ]:
!pip install torch
!pip install scipy numpy pandas fastdtw

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install tslearn

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import scipy.io as sio
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from torch.utils.data import Dataset, DataLoader
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import random
import json
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n DEVICE: {device}")
DATA_DIR = Path("/content/drive/MyDrive/Data")
TRAIN_DIR = DATA_DIR / "Training_data"
DEV_DIR = DATA_DIR / "Dev_data"
TEST_DIR = DATA_DIR / "Test_data"
SAVE_DIR = Path("/content/drive/MyDrive/huber_models")
SAVE_DIR.mkdir(exist_ok=True)
from tslearn.metrics import SoftDTW
N_POINTS = 100
EPOCHS = 100
BATCH_SIZE = 32
RESIDUAL_SCALES = {
'Contour1': 0.05,
'Contour2': 0.03
}
HYPERPARAMS = {
    'Contour1': {
        'lr': 2e-5,
        'weight_decay': 1e-5,
        'smooth_weight': 0.01
    },
    'Contour2': {
        'lr': 1e-5,
        'weight_decay': 1e-6,
        'smooth_weight': 0.02
    },
    'Contour3': {
        'lr': 1.5e-5,
        'weight_decay': 1e-5,
        'smooth_weight': 0.015
    }
}
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


 DEVICE: cuda


In [ ]:
def resample(pts,n=100):
  try:
    pts = np.array(pts,dtype=np.float32)
    if pts.ndim != 2:
      return None
    if pts.shape[0] == 2:
      pts = pts.T
    if pts.shape[1] != 2:
      return None
    if len(pts) < 2:
      return None
    diffs = np.diff(pts,axis=0)
    arc = np.concatenate([[0],np.cumsum(np.sqrt((diffs**2).sum(1)))])
    if arc[-1]==0:
      return None
    arc /=arc[-1]
    _,idx = np.unique(arc,return_index=True)
    arc = arc[idx]
    pts = pts[idx]
    if len(arc) < 2:
      return None
    t = np.linspace(0, 1, n)
    fx = interp1d(arc,pts[:,0],kind='linear',fill_value='extrapolate')
    fy = interp1d(arc,pts[:,1],kind='linear',fill_value='extrapolate')
    pts = np.stack([fx(t),fy(t)], axis=1)
    return pts.astype(np.float32)
  except:
    return None
def normalize_triplet(prev_pts,future_pts,gt_pts):
    all_pts = np.concatenate([
        prev_pts,
        future_pts,

    ], axis=0)

    mean = all_pts.mean(axis=0)
    std = all_pts.std(axis=0) + 1e-6
    prev_pts = (prev_pts - mean) / std
    future_pts = (future_pts - mean) / std
    gt_pts = (gt_pts - mean) / std
    return prev_pts,future_pts,gt_pts,mean,std
def augment_contour(pts):
  pts=pts.copy()
  angle = np.random.uniform(-5, 5) * np.pi / 180
  c = np.cos(angle)
  s = np.sin(angle)
  rot = np.array([
        [c, -s],
        [s, c]
    ])
  pts = pts@rot.T
  pts += np.random.uniform(-0.02,0.02,size=2)
  pts *= np.random.uniform(0.98,1.02)
  return pts
def load_contour(mat):
  ann = mat['annotation']
  c1,c2,c3=[],[],[]
  for i in range(len(ann)):
    c1.append(ann[i].contour1)
    c2.append(ann[i].contour2)
    c3.append(ann[i].contour3)
  return c1,c2,c3
def build_dataset(data_dir):
   sample_c1 = []
   sample_c2 = []
   sample_c3 = []
   files = sorted(list(Path(data_dir).glob("*.mat")))
   print(f"\n Found {len(files)} files")
   for j in range(len(files)):
     try:
        mat = sio.loadmat(files[j],squeeze_me = True,struct_as_record=False)
        c1,c2,c3 = load_contour(mat)
        total_frames = len(c1)
        for i in range(total_frames):
           max_length = min(i,total_frames-i-1)
           if max_length<=0:
            continue
           for n in range(1,max_length+1):
               contour_set =  [
                   (c1,sample_c1),
                   (c2,sample_c2),
                   (c3,sample_c3),
               ]
               for contour_data,samples in contour_set:
                  prev_pts = resample(
                        contour_data[i - n],
                        N_POINTS
                  )
                  future_pts = resample(
                        contour_data[i + n],
                        N_POINTS
                  )
                  gt_pts = resample(
                        contour_data[i],
                        N_POINTS
                  )
                  if prev_pts is None:
                        continue
                  if future_pts is None:
                        continue
                  if gt_pts is None:
                        continue
                  prev_pts,future_pts,gt_pts,mean,std = normalize_triplet(prev_pts,future_pts,gt_pts)
                  linear = (prev_pts + future_pts)/2.0
                  residual = gt_pts - linear
                  samples.append({'prev': prev_pts,
                        'future': future_pts,
                        'linear': linear,
                        'residual': residual,
                        'mean':mean,
                        'std':std
                       })
     except Exception as e:
       print(f"Error: {files[j].name}")
       print(e)
   return (
        sample_c1,
        sample_c2,
        sample_c3
    )
class ResidualDataset(Dataset):
  def __init__(self,samples,augment=False):
    self.samples = samples
    self.augment = augment
  def __len__(self):
    return len(self.samples)
  def __getitem__(self,idx):
    sample = self.samples[idx]
    prev_pts = sample['prev'].copy()
    future_pts = sample['future'].copy()
    linear = sample['linear'].copy()
    residual = sample['residual'].copy()
    mean = sample['mean'].copy()
    std = sample['std'].copy()
    if self.augment:
       prev_pts = augment_contour(prev_pts)
       future_pts = augment_contour(future_pts)
    return (torch.tensor(
                prev_pts,
                dtype=torch.float32
            ),
            torch.tensor(
                future_pts,
                dtype=torch.float32
            ),
            torch.tensor(
                linear,
                dtype=torch.float32
            ),
            torch.tensor(
                residual,
                dtype=torch.float32
            ),
            torch.tensor(
                mean,
                dtype=torch.float32
            ),
            torch.tensor(
                std,
                dtype=torch.float32
            ))

In [ ]:
class ModelforContour1(nn.Module):
  def __init__(self):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(400,256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.2),

        nn.Linear(256, 192),
        nn.BatchNorm1d(192),
        nn.ReLU(),
        nn.Dropout(0.15),



        nn.Linear(192, 128),

        nn.ReLU(),
        nn.Dropout(0.15),

        nn.Linear(128, 200),

    )


  def forward(self,prev,future):
    batch = prev.shape[0]
    x = torch.cat([
        prev.view(batch,-1),
        future.view(batch,-1)
    ],dim=1)
    residual = self.network(x)
    return residual.view(batch,N_POINTS,2)

In [ ]:
def train_model(train_samples,dev_samples,model_class,model_name,contour_name):
  print(f"\n Training {model_name}")
  hp = HYPERPARAMS[contour_name]
  train_loader = DataLoader(ResidualDataset(train_samples, augment=True),
        batch_size=BATCH_SIZE,
        shuffle=True)
  dev_loader = DataLoader(ResidualDataset(dev_samples),
        batch_size=BATCH_SIZE,
        shuffle=False)
  model = model_class().to(DEVICE)
  optimizer = torch.optim.Adam(
      model.parameters(),
      lr = hp['lr'],
      weight_decay = hp['weight_decay']
  )
  huber = nn.HuberLoss(delta=0.1)
  best_dev = 999999
  patience = 50           # ← Stop after 15 epochs of no improvement
  patience_counter = 0
  best_epoch = 0
  for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for (prev,future,linear,residual,mean,std) in train_loader:
      prev = prev.to(DEVICE)
      future = future.to(DEVICE)
      linear = linear.to(DEVICE)
      residual = residual.to(DEVICE)
      pred_residual = model(prev,future)
      if contour_name in ['Contour1','Contour2']:
        pred_final = linear + pred_residual*RESIDUAL_SCALES[contour_name]
      else:
        pred_final = linear + pred_residual
      gt_final = linear + residual
      regression_loss = huber(pred_final,gt_final)
      smooth_loss = (torch.diff(pred_final,dim=1)**2).mean()
      loss = regression_loss + hp['smooth_weight']*smooth_loss
      optimizer.zero_grad()
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
      optimizer.step()
      train_loss += loss.item()
    avg_train = train_loss/len(train_loader)
    model.eval()
    dev_loss = 0
    with torch.no_grad():
      for (prev,future,linear,residual,mean,std) in dev_loader:
        prev = prev.to(DEVICE)
        future = future.to(DEVICE)
        linear = linear.to(DEVICE)
        residual = residual.to(DEVICE)
        pred_residual = model(prev,future)
        if contour_name in ['Contour1','Contour2']:
          pred_final = linear + pred_residual*RESIDUAL_SCALES[contour_name]
        else:
          pred_final = linear + pred_residual
        gt_final = linear + residual
        loss = huber(pred_final,gt_final)
        dev_loss += loss.item()
      avg_dev = dev_loss/len(dev_loader)
      print(f"Epoch {epoch+1:03d} | "f"Train: {avg_train:.6f} | "f"Dev: {avg_dev:.6f}")
      if avg_dev < best_dev:
        patience_counter = 0
        best_dev = avg_dev
        torch.save(model.state_dict(),SAVE_DIR/f"{model_name}.pth")
        print("Saved")
      else:
        patience_counter += 1
        if patience_counter >= patience:
          print(f"Early stopping at epoch {epoch+1}")
          break

In [ ]:
def test_model(test_samples,model_class,model_name,contour_name):
  print(f"\n Testing {model_name}")
  pred_graph = []
  gt_graph = []
  model = model_class().to(DEVICE)
  model.load_state_dict(
    torch.load(
        SAVE_DIR / f"{model_name}.pth",
        map_location=DEVICE
    )
)
  test_loader = DataLoader(ResidualDataset(test_samples, augment=False),
        batch_size=BATCH_SIZE,
        shuffle=False)
  dtw_set = []
  model.eval()
  with torch.no_grad():
    for (prev,future,linear,residual,mean,std) in test_loader:
      prev = prev.to(DEVICE)
      future = future.to(DEVICE)
      linear = linear.to(DEVICE)
      residual = residual.to(DEVICE)
      pred_residual = model(prev,future)
      if contour_name in ['Contour1','Contour2']:
          pred_final = linear + pred_residual*RESIDUAL_SCALES[contour_name]
      else:
          pred_final = linear + pred_residual
      gt_final = linear + residual
      for i in range(len(pred_final)):
        score, _ = fastdtw(
        pred_final[i].cpu().numpy()*std[i].cpu().numpy() + mean[i].cpu().numpy(),
        gt_final[i].cpu().numpy()*std[i].cpu().numpy() + mean[i].cpu().numpy(),
        dist=euclidean
        )
        dtw_set.append(score)

  return dtw_set

In [ ]:
print("\nLoading data...")
train_c1, train_c2, train_c3 = build_dataset(TRAIN_DIR)
dev_c1, dev_c2, dev_c3 = build_dataset(DEV_DIR)
test_c1, test_c2, test_c3 = build_dataset(TEST_DIR)



Loading data...

 Found 54 files

 Found 4 files

 Found 4 files


In [ ]:
print("\nTRAINING")
train_model(
    train_c1,
    dev_c1,
    ModelforContour1,
    "contour1_huberfinal-7158",
    "Contour1"
)

print("\n HUBER TRAINING COMPLETE")


TRAINING

 Training contour1_huberfinal-7158
Epoch 001 | Train: 0.003032 | Dev: 0.002634
Saved
Epoch 002 | Train: 0.002885 | Dev: 0.002535
Saved
Epoch 003 | Train: 0.002839 | Dev: 0.002512
Saved
Epoch 004 | Train: 0.002812 | Dev: 0.002506
Saved
Epoch 005 | Train: 0.002792 | Dev: 0.002501
Saved
Epoch 006 | Train: 0.002776 | Dev: 0.002504
Epoch 007 | Train: 0.002766 | Dev: 0.002487
Saved
Epoch 008 | Train: 0.002757 | Dev: 0.002530
Epoch 009 | Train: 0.002748 | Dev: 0.002488
Epoch 010 | Train: 0.002744 | Dev: 0.002490
Epoch 011 | Train: 0.002741 | Dev: 0.002494
Epoch 012 | Train: 0.002738 | Dev: 0.002476
Saved
Epoch 013 | Train: 0.002737 | Dev: 0.002485
Epoch 014 | Train: 0.002737 | Dev: 0.002510
Epoch 015 | Train: 0.002735 | Dev: 0.002462
Saved
Epoch 016 | Train: 0.002735 | Dev: 0.002500
Epoch 017 | Train: 0.002731 | Dev: 0.002503
Epoch 018 | Train: 0.002733 | Dev: 0.002536
Epoch 019 | Train: 0.002731 | Dev: 0.002511
Epoch 020 | Train: 0.002732 | Dev: 0.002515
Epoch 021 | Train: 0.00273

In [ ]:
dtw3 = test_model(
    test_c1,
    ModelforContour1,
    "contour1_huberfinal-7158",
    "Contour1"
)



 Testing contour1_huberfinal-7158


In [ ]:
for i in range(len(dtw3)):
  print(f"DTW of contour3 : {dtw3[i]}")
import pandas as pd

# Assuming you already have these lists from your code:
# dtw_1, dtw_2, dtw_3, Avg_DTW, frameno_list, temporal_offset_list

# Create a DataFrame
df = pd.DataFrame({



    'DTW_Contour3': dtw3,

})

# Save to CSV
df.to_csv('dtw_resultscontour1final898.csv', index=False)

print(f"✅ CSV saved with {len(df)} rows")

DTW of contour3 : 33.00328526487602
DTW of contour3 : 44.27842912152224
DTW of contour3 : 54.00475929268636
DTW of contour3 : 38.44492085782708
DTW of contour3 : 38.10067053735519
DTW of contour3 : 57.168095741545315
DTW of contour3 : 51.887773623646446
DTW of contour3 : 56.799035370240716
DTW of contour3 : 59.24910504393918
DTW of contour3 : 52.74223066038661
DTW of contour3 : 32.93584069524024
DTW of contour3 : 52.28340933695834
DTW of contour3 : 54.388819404648295
DTW of contour3 : 61.643205895287124
DTW of contour3 : 71.32172121731011
DTW of contour3 : 62.21272082118302
DTW of contour3 : 69.10809325090726
DTW of contour3 : 75.86365253732588
DTW of contour3 : 88.86912699157452
DTW of contour3 : 77.9447950742026
DTW of contour3 : 75.49032626457219
DTW of contour3 : 28.15183530634512
DTW of contour3 : 48.60337301914004
DTW of contour3 : 59.337556651287095
DTW of contour3 : 60.576974018802524
DTW of contour3 : 56.835114405442425
DTW of contour3 : 62.8729527689839
DTW of contour3 : 49.8